<a href="https://colab.research.google.com/github/davidavni1221/Intro-to-AI-Final-Project/blob/main/Deep_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.datasets as dset
import torchvision.transforms as T
from torch.utils.data import DataLoader, sampler

# 1. Environment and Data Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Image normalization for CIFAR-10
transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Loading and splitting the dataset
NUM_TRAIN = 49000
cifar10_train = dset.CIFAR10('./datasets', train=True, download=True, transform=transform)
loader_train = DataLoader(cifar10_train, batch_size=64,
                          sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN)))

cifar10_val = dset.CIFAR10('./datasets', train=True, download=True, transform=transform)
loader_val = DataLoader(cifar10_val, batch_size=64,
                        sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN, 50000)))

# 2. Utility Class
class Flatten(nn.Module):
    def forward(self, x):
        return x.view(x.size(0), -1)

# 3. Model Validation Function
def validate_model(lr_value):
    print(f"Testing learning rate: {lr_value}")

    # Simple architecture for hyperparameter search
    temp_model = nn.Sequential(
        Flatten(),
        nn.Linear(3*32*32, 512),
        nn.ReLU(),
        nn.Linear(512, 10)
    ).to(device)

    optimizer = optim.Adam(temp_model.parameters(), lr=lr_value)

    # Short training loop for testing
    for e in range(2):
        temp_model.train()
        for x, y in loader_train:
            x, y = x.to(device), y.to(device)
            scores = temp_model(x)
            loss = F.cross_entropy(scores, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Evaluation phase
    temp_model.eval()
    num_correct, num_samples = 0, 0
    with torch.no_grad():
        for x, y in loader_val:
            x, y = x.to(device), y.to(device)
            scores = temp_model(x)
            _, preds = scores.max(1)
            num_correct += (preds == y).sum()
            num_samples += preds.size(0)

    return float(num_correct) / num_samples

# 4. Execution and Comparison
lr_list = [1e-3, 1e-4]
results = {}

for lr in lr_list:
    results[lr] = validate_model(lr)

# Final results display
print("\n--- Hyperparameter Selection Results ---")
for lr, acc in results.items():
    print(f"Learning Rate {lr}: Validation Accuracy = {acc*100:.2f}%")

Testing learning rate: 0.001
Testing learning rate: 0.0001

--- Hyperparameter Selection Results ---
Learning Rate 0.001: Validation Accuracy = 46.20%
Learning Rate 0.0001: Validation Accuracy = 49.80%


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, sampler
import torchvision.datasets as dset
import torchvision.transforms as T
import torch.nn.functional as F

# Device configuration (GPU if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# 1. CIFAR-10 Dataset Setup and Normalization
transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

NUM_TRAIN = 49000
# Training loader
cifar10_train = dset.CIFAR10('./datasets', train=True, download=True, transform=transform)
loader_train = DataLoader(cifar10_train, batch_size=64,
                          sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN)))

# Validation loader
cifar10_val = dset.CIFAR10('./datasets', train=True, download=True, transform=transform)
loader_val = DataLoader(cifar10_val, batch_size=64,
                        sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN, 50000)))

# Utility class for reshaping the output of conv layers
class Flatten(nn.Module):
    def forward(self, x):
        return x.view(x.size(0), -1)

# 2. CNN Architecture Definition
model = nn.Sequential(
    # First Conv Block
    nn.Conv2d(3, 32, kernel_size=3, padding=1),
    nn.BatchNorm2d(32),
    nn.ReLU(),
    nn.MaxPool2d(2, 2),

    # Second Conv Block
    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.MaxPool2d(2, 2),

    # Third Conv Block
    nn.Conv2d(64, 128, kernel_size=3, padding=1),
    nn.BatchNorm2d(128),
    nn.ReLU(),
    nn.MaxPool2d(2, 2),

    # Fully Connected Layers
    Flatten(),
    nn.Linear(128 * 4 * 4, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, 10)
).to(device)

# Optimization algorithm
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 3. Main Training Loop
def train_model(model, optimizer, epochs=10):
    for e in range(epochs):
        model.train()
        for t, (x, y) in enumerate(loader_train):
            x, y = x.to(device), y.to(device)

            # Forward pass
            scores = model(x)
            loss = F.cross_entropy(scores, y)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # Validation check at the end of each epoch
        model.eval()
        num_correct, num_samples = 0, 0
        with torch.no_grad():
            for x, y in loader_val:
                x, y = x.to(device), y.to(device)
                scores = model(x)
                _, preds = scores.max(1)
                num_correct += (preds == y).sum()
                num_samples += preds.size(0)

        acc = float(num_correct) / num_samples
        print(f'Epoch {e+1}, Loss: {loss.item():.4f}, Val Accuracy: {100 * acc:.2f}%')

# Start training process
train_model(model, optimizer)

Using device: cuda
Epoch 1, Loss: 1.0927, Val Accuracy: 66.00%
Epoch 2, Loss: 0.9945, Val Accuracy: 72.40%
Epoch 3, Loss: 1.1960, Val Accuracy: 73.50%
Epoch 4, Loss: 0.5616, Val Accuracy: 77.10%
Epoch 5, Loss: 0.4443, Val Accuracy: 79.60%
Epoch 6, Loss: 0.6295, Val Accuracy: 76.70%
Epoch 7, Loss: 0.2513, Val Accuracy: 79.40%
Epoch 8, Loss: 0.5303, Val Accuracy: 81.00%
Epoch 9, Loss: 0.3466, Val Accuracy: 78.20%
Epoch 10, Loss: 0.3419, Val Accuracy: 80.50%
